# FashionCNN: Optimizer Comparison with OneCycleLR

This self-contained Colab notebook loads and normalizes Fashion-MNIST, implements FashionCNN and a registry of 13 notebook-defined optimizers, runs learning-rate sweeps and seeded training runs, and saves metrics, plots, and a results table. The final cell can also package the outputs into a ZIP archive.

FashionCNN has three convolutional blocks with 32, 64, and 128 channels. Each block contains two Conv-BatchNorm-ReLU layers, max pooling, and Dropout2d(0.1). Global average pooling feeds a 128-to-128-to-10 classifier with ReLU and Dropout(0.4), for 304,682 trainable parameters. Inputs are normalized with mean 0.2860 and standard deviation 0.3530; no data augmentation is used. `FashionCNN()` defaults to native PyTorch AdamW, while the comparison runner selects methods from the notebook's optimizer registry. AdamW uses weight decay 1e-4.

In Colab, select **Runtime > Change runtime type > GPU**, then run the cells in order. The setup uses the PyTorch and torchvision versions available in the runtime and trains in float32. The default configuration is 30 epochs, 3 seeds, and batch size 128. Each learning-rate sweep trains for 2 epochs on the first 20,000 training examples and evaluates on the last 10,000 training examples; the subsets do not overlap. The sweep may not select the best learning rate for the full training budget, and each trial completes its own two-epoch OneCycleLR cycle.

OneCycleLR is used for every optimizer, with `max_lr = 10 * lr` and an initial rate of `0.4 * lr`; momentum cycling is disabled for the comparison. Final runs train on all 60,000 training examples and pass the official 10,000-image test set to Lightning's validation loop. Therefore, the saved `eval_loss` and `eval_acc` are test metrics, despite the validation-loop naming.

This notebook's protocol is not identical to the paper's: the paper uses a four-epoch sweep and native PyTorch implementations for eleven standard optimizers, whereas this self-contained notebook uses its own implementations for the comparison registry. The paper includes a sweep figure; the notebook also generates one.

Set `ONLY = ["AdamW"]` to run a subset. For a short trial, set `EPOCHS = 1`, `SEEDS = 1`, and `SWEEP_EPOCHS = 1`, and use an output directory separate from the full experiment. With `USE_DRIVE = True`, outputs are stored under `DRIVE_FOLDER` in Google Drive; otherwise they are written under `fashion_mnist_fashioncnn_outputs`. Resuming is supported at completed sweep-trial and training-run boundaries, not mid-epoch. Set `DOWNLOAD_ZIP = True` to download the output archive. The complete comparison may take several hours.

In [ ]:
%pip install -q "lightning>=2.6,<3" matplotlib numpy

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import time
import shutil
from pathlib import Path

import lightning.pytorch as pl
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display

print("PyTorch:", torch.__version__, "| Lightning:", pl.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
EPOCHS = 30
SEEDS = 3
BATCH = 128
SWEEP_EPOCHS = 4
SWEEP_SIZE = 20000
ONLY = None
USE_DRIVE = False
DRIVE_FOLDER = "fashion_mnist_fashioncnn_pytorch_custom_optimizers"
DOWNLOAD_ZIP = False

if min(EPOCHS, SEEDS, BATCH, SWEEP_EPOCHS, SWEEP_SIZE) < 1:
    raise ValueError("Epochs, seeds, batch size, sweep epochs, and sweep size must be positive.")
if SWEEP_SIZE > 50000:
    raise ValueError("The sweep must not overlap with the final 10,000 training examples reserved for validation.")

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_DIR = BASE_DIR / "data"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive") / DRIVE_FOLDER
else:
    OUTPUT_DIR = BASE_DIR / "fashion_mnist_fashioncnn_pytorch_custom_outputs"
RESULTS_DIR = OUTPUT_DIR / "results"
IMAGES_DIR = OUTPUT_DIR / "images"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_PATH = RESULTS_DIR / "results_fashioncnn.json"
print("Output:", OUTPUT_DIR)

In [ ]:
from torch.optim import Optimizer


class _CustomOptimizer(Optimizer):
    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is not None:
                    self._update(group, parameter, parameter.grad, self.state[parameter])
        return loss

    @staticmethod
    def _buffer(state, name, parameter):
        if name not in state:
            state[name] = torch.zeros_like(parameter)
        return state[name]


class AdaBelief(_CustomOptimizer):
    """Adaptive moments based on the gradient's deviation from its prediction."""

    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-16):
        super().__init__(params, dict(lr=lr, betas=betas, eps=eps))

    def _update(self, group, parameter, gradient, state):
        beta1, beta2 = group["betas"]
        step = state.get("step", 0) + 1
        state["step"] = step
        mean = self._buffer(state, "mean", parameter)
        variance = self._buffer(state, "variance", parameter)
        mean.lerp_(gradient, 1 - beta1)
        deviation = gradient - mean
        variance.mul_(beta2).addcmul_(deviation, deviation, value=1 - beta2)
        variance.add_(group["eps"])
        mean_hat = mean / (1 - beta1 ** step)
        variance_hat = variance / (1 - beta2 ** step)
        parameter.addcdiv_(mean_hat, variance_hat.sqrt() + group["eps"], value=-group["lr"])


class Lion(_CustomOptimizer):
    """Sign-based momentum optimizer with decoupled weight decay."""

    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.1):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    def _update(self, group, parameter, gradient, state):
        beta1, beta2 = group["betas"]
        momentum = self._buffer(state, "momentum", parameter)
        update = torch.lerp(momentum, gradient, 1 - beta1)
        parameter.mul_(1 - group["lr"] * group["weight_decay"])
        parameter.add_(update.sign(), alpha=-group["lr"])
        momentum.lerp_(gradient, 1 - beta2)


_REQUESTED_ONLY = ONLY
ONLY = None

In [ ]:
import torchmetrics

OPTIMIZERS = {
    "SGD": (lambda params, lr: torch.optim.SGD(params, lr=lr), [0.01, 0.03, 0.1]),
    "Momentum": (lambda params, lr: torch.optim.SGD(params, lr=lr, momentum=0.9), [0.003, 0.01, 0.03]),
    "Nesterov": (lambda params, lr: torch.optim.SGD(params, lr=lr, momentum=0.9, nesterov=True), [0.003, 0.01, 0.03]),
    "AdaGrad": (lambda params, lr: torch.optim.Adagrad(params, lr=lr), [0.01, 0.03, 0.1]),
    "AdaDelta": (lambda params, lr: torch.optim.Adadelta(params, lr=lr), [0.3, 1.0, 3.0]),
    "RMSProp": (lambda params, lr: torch.optim.RMSprop(params, lr=lr), [1e-4, 3e-4, 1e-3]),
    "Adam": (lambda params, lr: torch.optim.Adam(params, lr=lr), [3e-4, 1e-3, 3e-3]),
    "AMSGrad": (lambda params, lr: torch.optim.Adam(params, lr=lr, amsgrad=True), [3e-4, 1e-3, 3e-3]),
    "AdamW": (lambda params, lr: torch.optim.AdamW(params, lr=lr, weight_decay=1e-4), [3e-4, 1e-3, 3e-3]),
    "Nadam": (lambda params, lr: torch.optim.NAdam(params, lr=lr), [3e-4, 1e-3, 3e-3]),
    "RAdam": (lambda params, lr: torch.optim.RAdam(params, lr=lr), [3e-4, 1e-3, 3e-3]),
}

MODEL_CONFIG = {
    "model": "FashionCNN-v1-pytorch-optimizers",
    "scheduler": "OneCycleLR",
    "optimizer_implementation": "torch.optim",
    "max_lr_factor": 10,
    "cycle_momentum": False,
    "adamw_weight_decay": 1e-4,
}


def model_config():
    return dict(MODEL_CONFIG)


def conv_block(c_in, c_out):
    """Two Conv-BN-ReLU layers followed by pooling and spatial dropout."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
        nn.Dropout2d(0.1),
    )


class FashionCNN(pl.LightningModule):
    """Fashion-MNIST CNN trained with OneCycleLR and native PyTorch optimizers."""

    def __init__(self, lr=1e-3, weight_decay=1e-4, num_classes=10,
                 max_epochs=30, optimizer_name=None):
        super().__init__()
        self.save_hyperparameters()
        self.features = nn.Sequential(
            conv_block(1, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )
        self.train_acc = torchmetrics.Accuracy("multiclass", num_classes=num_classes)
        self.val_acc = torchmetrics.Accuracy("multiclass", num_classes=num_classes)
        self.test_acc = torchmetrics.Accuracy("multiclass", num_classes=num_classes)
        self.test_cm = torchmetrics.ConfusionMatrix("multiclass", num_classes=num_classes)
        self.iter_losses = []
        self.train_losses = []
        self.eval_losses = []
        self.eval_accs = []
        self._train_loss_sum = 0.0
        self._train_example_count = 0
        self._eval_loss_sum = 0.0
        self._eval_correct_count = 0
        self._eval_example_count = 0

    def forward(self, images):
        return self.classifier(self.features(images))

    def _step(self, batch):
        images, targets = batch
        logits = self(images)
        return F.cross_entropy(logits, targets), logits, targets

    def on_train_epoch_start(self):
        self._train_loss_sum = 0.0
        self._train_example_count = 0

    def training_step(self, batch, batch_idx):
        loss, logits, targets = self._step(batch)
        batch_size = targets.size(0)
        loss_value = loss.detach().item()
        self.iter_losses.append(loss_value)
        self._train_loss_sum += loss_value * batch_size
        self._train_example_count += batch_size
        self.train_acc.update(logits, targets)
        self.log("train/loss", loss, on_epoch=True, prog_bar=True, batch_size=batch_size)
        self.log("train/acc", self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        if self._train_example_count:
            self.train_losses.append(self._train_loss_sum / self._train_example_count)

    def on_validation_epoch_start(self):
        self._eval_loss_sum = 0.0
        self._eval_correct_count = 0
        self._eval_example_count = 0

    def validation_step(self, batch, batch_idx):
        loss, logits, targets = self._step(batch)
        batch_size = targets.size(0)
        self._eval_loss_sum += loss.detach().item() * batch_size
        self._eval_correct_count += (logits.argmax(1) == targets).sum().item()
        self._eval_example_count += batch_size
        self.val_acc.update(logits, targets)
        self.log("val/loss", loss, prog_bar=True, batch_size=batch_size)
        self.log("val/acc", self.val_acc, prog_bar=True)

    def on_validation_epoch_end(self):
        if self._eval_example_count and not self.trainer.sanity_checking:
            self.eval_losses.append(self._eval_loss_sum / self._eval_example_count)
            self.eval_accs.append(self._eval_correct_count / self._eval_example_count)

    def on_test_epoch_start(self):
        self.test_cm.reset()

    def test_step(self, batch, batch_idx):
        loss, logits, targets = self._step(batch)
        self.test_acc.update(logits, targets)
        self.test_cm.update(logits.argmax(1), targets)
        self.log("test/loss", loss, batch_size=targets.size(0))
        self.log("test/acc", self.test_acc)

    def predict_step(self, batch, batch_idx):
        images = batch[0] if isinstance(batch, (list, tuple)) else batch
        return self(images).softmax(dim=1)

    def configure_optimizers(self):
        if self.hparams.optimizer_name is None:
            optimizer = torch.optim.AdamW(
                self.parameters(), lr=self.hparams.lr,
                weight_decay=self.hparams.weight_decay,
            )
        else:
            factory, _ = OPTIMIZERS[self.hparams.optimizer_name]
            optimizer = factory(self.parameters(), self.hparams.lr)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=self.hparams.lr * 10,
            total_steps=self.trainer.estimated_stepping_batches,
            cycle_momentum=False,
        )
        return {"optimizer": optimizer,
                "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}


CNN = FashionCNN

MEAN, STD = 0.2860, 0.3530
CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]


def load_fashion_mnist(root="data"):
    train_data = datasets.FashionMNIST(str(root), train=True, download=True)
    test_data = datasets.FashionMNIST(str(root), train=False, download=True)
    return train_data.data, train_data.targets, test_data.data, test_data.targets


def preprocess(images):
    return ((images.float() / 255 - MEAN) / STD).unsqueeze(1)


class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, train_images, train_targets, eval_images, eval_targets,
                 batch_size=128, seed=0, eval_batch_size=2000):
        super().__init__()
        self.train_images = train_images
        self.train_targets = train_targets
        self.eval_images = eval_images
        self.eval_targets = eval_targets
        self.batch_size = batch_size
        self.seed = seed
        self.eval_batch_size = eval_batch_size

    def setup(self, stage=None):
        self.train_dataset = TensorDataset(self.train_images, self.train_targets)
        self.eval_dataset = TensorDataset(self.eval_images, self.eval_targets)

    def train_dataloader(self):
        generator = torch.Generator().manual_seed(self.seed)
        return DataLoader(self.train_dataset, batch_size=self.batch_size,
                          shuffle=True, generator=generator)

    def val_dataloader(self):
        return DataLoader(self.eval_dataset, batch_size=self.eval_batch_size, shuffle=False)


names = list(OPTIMIZERS) if ONLY is None else list(ONLY)
if not names or len(names) != len(set(names)) or any(name not in OPTIMIZERS for name in names):
    raise ValueError("ONLY must contain unique names from the PyTorch optimizer registry.")
print("PyTorch optimizers:", ", ".join(names))

In [ ]:
OPTIMIZERS.update({
    "AdaBelief": (lambda params, lr: AdaBelief(params, lr=lr), [3e-4, 1e-3, 3e-3]),
    "Lion": (lambda params, lr: Lion(params, lr=lr, weight_decay=0.1), [3e-5, 1e-4, 3e-4]),
})

MODEL_CONFIG.update({
    "model": "FashionCNN-v1-pytorch-custom-optimizers",
    "optimizer_implementation": "torch.optim plus notebook-defined AdaBelief and Lion",
    "adabelief_eps": 1e-16,
    "lion_weight_decay": 0.1,
})

names = list(OPTIMIZERS) if _REQUESTED_ONLY is None else list(_REQUESTED_ONLY)
if not names or len(names) != len(set(names)) or any(name not in OPTIMIZERS for name in names):
    raise ValueError("ONLY must contain unique names from the optimizer registry.")
print("Optimizers:", ", ".join(names))

In [ ]:
xtr, ytr, xte, yte = load_fashion_mnist(DATA_DIR)
assert tuple(xtr.shape) == (60000, 28, 28)
assert tuple(xte.shape) == (10000, 28, 28)
assert len(ytr) == 60000 and len(yte) == 10000
Xtr, Xte = preprocess(xtr), preprocess(xte)
print("Train:", tuple(Xtr.shape), "| Test:", tuple(Xte.shape))

plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3})
fig, axes = plt.subplots(2, 5, figsize=(8, 3.6))
for class_index, axis in enumerate(axes.ravel()):
    sample_index = int((ytr == class_index).nonzero()[0, 0])
    axis.imshow(xtr[sample_index], cmap="gray")
    axis.set_title(CLASSES[class_index], fontsize=8)
    axis.axis("off")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "fmnist_samples.png", dpi=180)
plt.close(fig)
display(Image(filename=str(IMAGES_DIR / "fmnist_samples.png")))

In [ ]:
def train(name, Xtr, ytr, Xev, yev, epochs, seed, batch=128, lr=None):
    lr = lr if lr is not None else 1e-3
    torch.manual_seed(seed)
    model = FashionCNN(optimizer_name=name, lr=lr, max_epochs=epochs)
    data = FashionMNISTDataModule(Xtr, ytr, Xev, yev, batch_size=batch, seed=seed)
    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator="auto",
        devices=1,
        precision="32-true",
        logger=False,
        enable_checkpointing=False,
        enable_model_summary=False,
        enable_progress_bar=False,
        deterministic=True,
        num_sanity_val_steps=0,
    )
    trainer.fit(model, datamodule=data)
    return {
        "train_loss": list(model.train_losses),
        "eval_loss": list(model.eval_losses),
        "eval_acc": list(model.eval_accs),
        "iter_loss": list(model.iter_losses),
    }


def save(res, path):
    path = Path(path)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w") as result_file:
        json.dump(res, result_file)
    os.replace(temporary, path)


config = dict(epochs=EPOCHS, seeds=SEEDS, batch=BATCH,
              sweep_epochs=SWEEP_EPOCHS, sweep_size=SWEEP_SIZE,
              **model_config())
if RESULTS_PATH.exists():
    with RESULTS_PATH.open() as result_file:
        res = json.load(result_file)
    if res.get("config") != config:
        raise ValueError("Saved results use a different model or configuration; choose a separate output directory.")
    print("Resuming from:", RESULTS_PATH)
else:
    res = {"config": config, "sweep": {}, "best_lr": {}, "runs": {}}
res.setdefault("sweep", {})
res.setdefault("best_lr", {})
res.setdefault("runs", {})


def sweep_one(name, lr):
    trials = res["sweep"].setdefault(name, {})
    if repr(lr) in trials:
        return
    started = time.time()
    history = train(name, Xtr[:SWEEP_SIZE], ytr[:SWEEP_SIZE],
                    Xtr[-10000:], ytr[-10000:], SWEEP_EPOCHS, seed=123,
                    batch=BATCH, lr=lr)
    value = history["eval_loss"][-1]
    trials[repr(lr)] = float(value) if np.isfinite(value) else None
    save(res, RESULTS_PATH)
    print(f"[sweep] {name:9s} lr={lr:<8g} val_loss={value:.4f} ({time.time()-started:.0f}s)", flush=True)


for name in names:
    if name in res["best_lr"]:
        continue
    _, grid = OPTIMIZERS[name]
    for lr in grid:
        sweep_one(name, lr)
    completed_extensions = len(res["sweep"][name]) - len(grid)
    for extension in range(completed_extensions, 3):
        valid = {float(key): value for key, value in res["sweep"][name].items() if value is not None}
        if not valid:
            break
        best = min(valid, key=valid.get)
        low = min(res["sweep"][name], key=float)
        high = max(res["sweep"][name], key=float)
        if best == float(high):
            sweep_one(name, float(high) * 3)
        elif best == float(low):
            sweep_one(name, float(low) / 3)
        else:
            break
    valid = {float(key): value for key, value in res["sweep"][name].items() if value is not None}
    if not valid:
        raise RuntimeError(f"All learning-rate trials diverged for {name}.")
    res["best_lr"][name] = min(valid, key=valid.get)
    save(res, RESULTS_PATH)
print("Selected learning rates:", {name: res["best_lr"][name] for name in names})

In [ ]:
for name in names:
    lr = res["best_lr"][name]
    runs = res["runs"].setdefault(name, [])
    for seed in range(len(runs), SEEDS):
        started = time.time()
        history = train(name, Xtr, ytr, Xte, yte, EPOCHS, seed=seed, batch=BATCH, lr=lr)
        history["seed"] = seed
        runs.append(history)
        save(res, RESULTS_PATH)
        print(f"[train] {name:9s} lr={lr:<7g} seed={seed} "
              f"test_acc={history['eval_acc'][-1]:.4f} ({time.time()-started:.0f}s)", flush=True)
print("Saved results:", RESULTS_PATH)

In [ ]:
with RESULTS_PATH.open() as result_file:
    res = json.load(result_file)

GROUPS = [
    ("SGD and adaptive methods",
     ["SGD", "Momentum", "Nesterov", "AdaGrad", "AdaDelta", "RMSProp"]),
    ("Adam-family, AdaBelief, and Lion",
     ["Adam", "AMSGrad", "AdamW", "Nadam", "RAdam", "AdaBelief", "Lion"]),
]
plot_names = [name for _, group in GROUPS for name in group
              if name in res["runs"] and res["runs"][name]]
if not plot_names:
    raise RuntimeError("Run the training cells before generating plots.")
cmap = plt.get_cmap("tab20")
COLORS = {name: cmap(index) for index, name in enumerate(
    name for _, group in GROUPS for name in group)}
plt.rcParams.update({"font.size": 9, "axes.grid": True, "grid.alpha": 0.3})


def stack(name, key):
    return np.array([run[key] for run in res["runs"][name]])


def seed_std(values):
    return values.std(axis=0, ddof=1) if len(values) > 1 else np.zeros_like(values[0])


def two_panel(filename, key, ylabel, logy=False, ylim=None, smooth=None, upto=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    for axis, (title, group) in zip(axes, GROUPS):
        for name in group:
            if name not in plot_names:
                continue
            values = stack(name, key)
            if upto:
                values = values[:, :upto]
            window = min(smooth, values.shape[1]) if smooth else None
            if window:
                kernel = np.ones(window) / window
                values = np.array([np.convolve(run, kernel, mode="valid") for run in values])
            mean, std = values.mean(0), seed_std(values)
            positions = (np.arange(1, len(mean) + 1) if not window
                         else np.arange(window, window + len(mean)))
            axis.plot(positions, mean, label=name, color=COLORS[name], lw=1.5)
            axis.fill_between(positions, mean - std, mean + std,
                              color=COLORS[name], alpha=0.15)
        axis.set_title(title)
        if axis.lines:
            axis.legend(fontsize=8)
        axis.set_xlabel("iteration" if smooth else "epoch")
        axis.set_ylabel(ylabel)
        if logy:
            axis.set_yscale("log")
        if ylim:
            axis.set_ylim(*ylim)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / f"{filename}.png", dpi=180)
    plt.close(fig)


sweep_names = [name for name in res["sweep"]
               if res["sweep"][name] and name in res["best_lr"]]
cols = 5
rows = int(np.ceil(len(sweep_names) / cols))
if rows:
    fig, axes = plt.subplots(rows, cols, figsize=(11, 2.3 * rows), squeeze=False)
    for axis, name in zip(axes.ravel(), sweep_names):
        points = sorted((float(key), value)
                        for key, value in res["sweep"][name].items()
                        if value is not None)
        learning_rates, losses = zip(*points)
        axis.semilogx(learning_rates, losses, "o-",
                      color=COLORS.get(name, "k"), ms=4)
        best = res["best_lr"][name]
        axis.plot(best, dict(points)[best], "k*", ms=11, zorder=5)
        axis.set_title(name, fontsize=9)
        axis.set_xlabel("learning rate")
    for axis in axes.ravel()[len(sweep_names):]:
        axis.axis("off")
    for row_index in range(min(2, rows)):
        axes[row_index, 0].set_ylabel("validation loss")
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "fmnist_lr_sweep.png", dpi=180)
    plt.close(fig)

two_panel("fmnist_train_loss", "train_loss", "training loss", logy=True)
accuracies = np.concatenate([stack(name, "eval_acc").ravel()
                             for name in plot_names])
accuracy_low = max(0.80, accuracies.min() - 0.01)
if accuracies.min() < 0.80:
    accuracy_low = max(0.0, accuracies.min() - 0.01)
two_panel("fmnist_test_acc", "eval_acc", "test accuracy",
          ylim=(accuracy_low, accuracies.max() + 0.005))
two_panel("fmnist_iter_loss", "iter_loss", "training loss (moving average)",
          logy=True, smooth=25, upto=2 * 469)

final = {name: stack(name, "eval_acc")[:, -1] for name in plot_names}
order = sorted(plot_names, key=lambda name: final[name].mean())
fig, axis = plt.subplots(figsize=(7, 4.2))
axis.barh(order, [100 * final[name].mean() for name in order],
          xerr=[100 * seed_std(final[name]) for name in order],
          color=[COLORS[name] for name in order], capsize=3)
low = 100 * min(final[name].mean() for name in order)
axis.set_xlim(low - 1.5, 100 * max(final[name].mean() for name in order) + 0.8)
for index, name in enumerate(order):
    axis.text(100 * final[name].mean() + 0.1 + 100 * seed_std(final[name]), index,
              f"{100 * final[name].mean():.2f}", va="center", fontsize=8)
axis.set_xlabel("final test accuracy (%)")
plt.tight_layout()
plt.savefig(IMAGES_DIR / "fmnist_final_acc.png", dpi=180)
plt.close(fig)

table_rows = []
for name in sorted(plot_names, key=lambda name: -final[name].mean()):
    train_losses = stack(name, "train_loss")[:, -1]
    eval_losses = stack(name, "eval_loss")[:, -1]
    table_rows.append(
        f"{name} & {res['best_lr'][name]:.3g} & {train_losses.mean():.3f} & "
        f"{eval_losses.mean():.3f} & {100 * final[name].mean():.2f} "
        f"$\\pm$ {100 * seed_std(final[name]):.2f} \\\\"
    )
(RESULTS_DIR / "table.tex").write_text("\n".join(table_rows) + "\n")
print("\n".join(table_rows))
for filename in ["fmnist_samples", "fmnist_lr_sweep", "fmnist_train_loss",
                 "fmnist_test_acc", "fmnist_iter_loss", "fmnist_final_acc"]:
    image_path = IMAGES_DIR / f"{filename}.png"
    if image_path.exists():
        display(Image(filename=str(image_path)))

In [ ]:
archive = shutil.make_archive(str(BASE_DIR / "fashion_mnist_fashioncnn_pytorch_results"), "zip", root_dir=OUTPUT_DIR)
print("Archive:", archive)
print(f"Contents: results/{RESULTS_PATH.name}, results/table.tex, and images/*.png")
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(archive)